## Overview

ベースラインを作成していく！
モデル：lightGBM（とりあえず）
特徴量：`daily_screentime, social_media_hours, work_study_hours, weekend_screen_hours`

In [114]:
import pandas as pd
import numpy as np

train_df = pd.read_csv('../data/train.csv')
test_df = pd.read_csv('../data/test.csv')

In [115]:
data = pd.concat([train_df, test_df])

In [116]:
data.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1.0
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0.0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0.0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1.0
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1.0


In [117]:
data['gender'].value_counts()

gender
Male      319019
Female    315890
Other     309516
Name: count, dtype: int64

In [118]:
data['stress_level'].value_counts()

stress_level
High      316495
Low       298529
Medium    297873
Name: count, dtype: int64

In [119]:
# stress_levelを数値に変換
data['stress_level'] = data['stress_level'].map({
  'Low': 0,
  'Medium': 1,
  'High': 2
})

data['stress_level'].value_counts(dropna=False)

stress_level
2.0    316495
0.0    298529
1.0    297873
NaN     74774
Name: count, dtype: int64

In [120]:
# genderを数値に変換
data['gender'] = data['gender'].map({
  'Male': 0,
  'Female': 1,
  'Other': 2
})

data.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,0.0,1.0,No,1.0
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,1.0,1.0,No,0.0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,1.0,0.0,Yes,0.0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,2.0,0.0,NaN,1.0
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,1.0,1.0,No,1.0


In [121]:
# TrainとTest分離
train_df = data.iloc[:len(train_df)].copy()
test_df = data.iloc[len(train_df):].copy()

from lightgbm import LGBMClassifier

# lightGBM
params = {
    "boosting_type": "gbdt",
    "objective": "binary",
    "metric": "binary_error",
    "num_iterations": 1000,
    "learning_rate": 0.02,
    "num_leaves": 16,
    "max_depth": -1,
    "min_data_in_leaf": 20,
    "min_sum_hessian_in_leaf": 1e-3,
    "bagging_fraction": 0.9,
    "bagging_freq": 1,
    "feature_fraction": 0.9,
    "lambda_l1": 0.0,
    "lambda_l2": 0.0,
    "random_state": 42,
    "verbosity": -1
}

lgbm = LGBMClassifier(**params)

In [122]:
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import StratifiedKFold

features = ['daily_screen_time_hours', 'social_media_hours', 'work_study_hours', 'weekend_screen_time']

X = train_df[features]
y = train_df['addicted_label']

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

auc_lgbm = cross_val_score(
  lgbm,
  X,
  y,
  cv=cv,
  scoring="roc_auc"
)

float(round(auc_lgbm.mean() * 100, 2))

93.88

AUCのときは`predict`じゃなくて`predict_proba`を使うぞ。

In [123]:
# 提出ファイル作成
lgbm.fit(X, y)
test_pred = lgbm.predict_proba(test_df[features])[:, 1]

submission = pd.DataFrame({
  "id": test_df['id'],
  "addicted_label": test_pred
})

In [124]:
submission.to_csv('../submissions/submission.csv', index=False)

In [126]:
submission.head()

,id,addicted_label
0,691369,0.992869
1,691370,0.924849
2,691371,0.982046
3,691372,0.978216
4,691373,0.993519


In [129]:
submission.shape

(296302, 2)

In [130]:
submission.isna().sum()

id                0
addicted_label    0
dtype: int64